# Лекција 13 - Меморија агента


## Подешавање

Овај бележник демонстрира како изградити агента за резервацију путовања са **постојећом меморијом** користећи **Microsoft Agent Framework** (MAF).

Учићете како различити типови агенцијске меморије — радна, краткорочна и дугорочна — утичу на то како агент задржава и користи информације током разговора.

**Претпоставке:**
- Microsoft Foundry пројекат са распоређеним моделом за ћаскање (нпр. `gpt-5-mini`).
- Пријављени преко Azure CLI — покрените `az login` у вашем терминалу.
- `AZURE_AI_PROJECT_ENDPOINT` — крајња тачка вашег Microsoft Foundry пројекта.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — назив вашег распоређеног модела.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import json
import dotenv
from typing import Annotated
from datetime import datetime

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## Врсте меморије агента

AI агенти могу користити различите врсте меморије, свака са својом посебном сврхом:

### Радна меморија
Сам разговор — поруке размењене у једној сесији. Агент може да се осврне на раније поруке у истом разговору како би одржао кохеренцију. У MAF-у се ово креира преко **`agent.create_session()`**, која враћа `AgentSession`.

### Краткорочна меморија
Информације које трају током трајања задатка или сесије, али се не чувају трајно. На пример, агент може да сакупља чињенице током вишекратног планирања у разговору и користи их да произведе коначни итинерер.

### Дугорочна меморија
Преференције и чињенице које трају **преко сесија**. Повратни корисник не би требало да понавља своје дијеталне рестрикције или стил путовања. Дугорочна меморија је обично подржана спољним складиштем — базом података, фајлом или векторским индексом — и доступна агенту преко алата.


## Радна меморија са сесијама

Најједноставнији облик меморије је сесија разговора. Када проследите исту сесију (креирану преко `agent.create_session()`) узастопним позивима `agent.run()`, агент види целу историју тог разговора и може да се сети претходних детаља.

Хајде да направимо агента за путовања и прикажемо радну меморију.


In [ ]:
agent = client.as_agent(
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

Агент је правилно запамтио буџет јер обе поруке деле исту сесију. Ово је **радна меморија** — постоји само током трајања сесије.

### Шта се дешава са новом нитима?

Ако креирамо **нову** сесију, агент нема меморију претходног разговора:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## Образац дугорочне меморије

Да бисмо памтили корисничке преференције **преко сесија**, потребно нам је трајно складиште које постоји ван нити разговора. Агенат приступа том складишту преко **алата** — функција које може позвати да сачува и преузме информације.

Испод имплементирамо једноставан у-памћењу продавницу преференција (у производњи бисте ово подржали базом података или векторским индексом) и излажемо је као алате које агент може користити.

### Архитектура
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### Сценарио 1 — Корисник који резервише путовање за годишњицу по први пут

Сара први пут посећује. Агента треба да сачува њене преференце помоћу алата и да их користи за препоруку хотела.


In [ ]:
travel_agent = client.as_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### Сценарио 2 — Сара се враћа недељама касније

Сара започиње **потпуно нову нит** (симулирајући нову сесију). Радна меморија је празна, али дугорочна продавница преференција и даље има њене информације. Агенс треба да их преузме и искористи за персонализацију препорука.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## Резиме

У овој лекцији сте научили три типа меморије агента и како их имплементирати помоћу Microsoft Agent Framework-а:

| Тип меморије | Механизам MAF | Век трајања |
|---|---|---|
| **Радна** | `agent.create_session()` | Један разговор |
| **Краткорочна** | Акомулирани контекст унутар нити | Један задатак / сесија |
| **Дугорочна** | Спољни извор приступан преко `@tool` функција | Кроз више сесија |

### Кључни закључци
1. **`agent.create_session()`** обезбеђује радну меморију — агент види пуну историју разговора у оквиру сесије.
2. **Нове сесије губе контекст** — без дугорочне меморије агент не може да се сети претходних разговора.
3. **`@tool` функције** премошћују јаз — оне омогућавају агенту да сачува и преузме информације из трајног извора.
4. **Персонализација се побољшава временом** — што је више преференција сачувано, боље су агентове препоруке.

### Примене у реалном свету
- **Корисничка подршка**: Памћење историје и преференција корисника
- **Персонални асистенти**: Одржавање контекста кроз дане или недеље
- **Здравство**: Прати информације и преференције пацијената
- **Е-трговина**: Персонализована куповина базирана на историји

### Следећи кораци
- Замена речника у меморији базом података или векторском продавницом (нпр. Azure AI Search)
- Додавање истека меморије за временски осетљиве информације
- Изградња мулти-агентских система са заједничком меморијом
- Истражите Cognee нотебук за меморију подржану графом знања


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Изјава о одрицању одговорности**:
Овај документ је преведен коришћењем услуге за аутоматски превод [Co-op Translator](https://github.com/Azure/co-op-translator). Иако тежимо тачности, имајте у виду да аутоматски преводи могу садржати грешке или нетачности. Оригинални документ на његовом изворном језику треба сматрати ауторитативним извором. За критичне информације препоручује се професионални људски превод. Нисмо одговорни за било каква неспоразума или погрешна тумачења која произилазе из коришћења овог превода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
